# FDSA Real Benchmark V2_U2 — Corrected Filter + Honest Scope

**Changes from V2_U1, and why:**

1. **Cell 6 filter is now actually wired to a real, unit-consistent threshold** — V2_U1's
   `FDSADriftFilter` built a `VectorizedFDSAPruner` but never called it; it applied a separate,
   disconnected hardcoded `-3.5` cutoff instead. Fixed here: the filter now uses a threshold
   relative to each step's own max logit (unit-consistent, actually capable of binding).

2. **Scope corrected: this notebook tests accuracy impact (EM/F1), not speed.** Masking logits
   to `-inf` via `jnp.where` does not shrink the array HF's `logits_processor` operates on —
   under JIT/XLA the compute graph has fixed shape regardless of how many entries are `-inf`.
   **This architecture cannot show a genuine compute-level speedup.** Any timing difference
   observed is JIT warm-up noise, not the mechanism working. Timing is still reported below,
   but explicitly labeled as non-evidentiary, with warm-up controlled for so the *noise floor*
   itself is at least fair.

3. **Explicit JIT warm-up added for both configurations before any timing starts** — this
   removes the run-order confound directly instead of just diagnosing it.

4. **Raw sample predictions printed side-by-side** — so identical EM/F1 can be visually
   confirmed as identical text, not just identical scores.


## 1–4. Environment / model / dataset / scoring — unchanged from V2_U1, re-run as before

In [ ]:
# Re-run your existing V2_U1 Cells 1-4, 8, 10 (device check, installs, model load, dataset, scoring fns)
# unchanged — omitted here to avoid duplicating what already works. Paste them above this cell.


## 5. Explicit warm-up phase — run BEFORE any timing

This compiles both code paths once each, on throwaway data, so the timed runs below never pay
first-call XLA compilation cost. This is the actual fix for the run-order confound.


In [ ]:
from transformers import FlaxLogitsProcessor, FlaxLogitsProcessorList
import jax.numpy as jnp
import jax

class RelativeThresholdFDSAFilter(FlaxLogitsProcessor):
    """
    Corrected filter: threshold is relative to each step's own max logit (unit-consistent),
    not an arbitrary absolute constant disconnected from the model's logit scale.
    margin is a plain, explicitly-labeled tunable hyperparameter — NOT claimed to be derived
    from D_limit/fractal dimension, since mixing those units was the V2_U1 bug. Tying this
    rigorously to the theory's contraction factor remains open work, not claimed here.
    """
    def __init__(self, margin: float = 5.0):
        self.margin = margin  # try 2.0 (aggressive), 5.0 (moderate), 10.0 (loose) across runs

    def __call__(self, input_ids: jnp.ndarray, scores: jnp.ndarray, cur_len: int) -> jnp.ndarray:
        max_score = jnp.max(scores, axis=-1, keepdims=True)
        cutoff = max_score - self.margin
        return jnp.where(scores < cutoff, -jnp.inf, scores)

def make_kwargs(margin=None):
    if margin is None:
        return dict(max_new_tokens=16, num_beams=4, do_sample=False)
    proc = FlaxLogitsProcessorList([RelativeThresholdFDSAFilter(margin=margin)])
    return dict(max_new_tokens=16, num_beams=4, do_sample=False, logits_processor=proc)

# --- Warm-up: run both configs once on a tiny throwaway batch, discard results ---
warmup_questions = questions[:2]
print("Warming up baseline JIT compilation...")
_ = run_generation(warmup_questions, make_kwargs(margin=None))
print("Warming up FDSA JIT compilation...")
_ = run_generation(warmup_questions, make_kwargs(margin=5.0))
print("Warm-up complete. Both code paths compiled. Timing below is now fair.")


## 6. Timed runs — post warm-up

In [ ]:
baseline_preds, baseline_time, baseline_tps = run_generation(questions, make_kwargs(margin=None))
baseline_em = sum(exact_match(p, ex["answers"]) for p, ex in zip(baseline_preds, eval_set)) / len(eval_set)
baseline_f1 = sum(f1_score(p, ex["answers"]) for p, ex in zip(baseline_preds, eval_set)) / len(eval_set)

fdsa_preds, fdsa_time, fdsa_tps = run_generation(questions, make_kwargs(margin=5.0))
fdsa_em = sum(exact_match(p, ex["answers"]) for p, ex in zip(fdsa_preds, eval_set)) / len(eval_set)
fdsa_f1 = sum(f1_score(p, ex["answers"]) for p, ex in zip(fdsa_preds, eval_set)) / len(eval_set)

print("="*70)
print("BASELINE (post warm-up) — EM: {:.4f} | F1: {:.4f} | Time: {:.2f}s | TPS: {:.1f}".format(
    baseline_em, baseline_f1, baseline_time, baseline_tps))
print("FDSA margin=5.0 (post warm-up) — EM: {:.4f} | F1: {:.4f} | Time: {:.2f}s | TPS: {:.1f}".format(
    fdsa_em, fdsa_f1, fdsa_time, fdsa_tps))
print("="*70)
print("NOTE: per Finding 6, any timing delta above is warm-up/measurement noise, NOT a")
print("validated compute-level speedup — masking cannot reduce FLOPs at this hook point.")
print("The number that means something here is Delta EM / Delta F1, not the timing.")
print("Delta EM: {:+.4f}   Delta F1: {:+.4f}".format(fdsa_em - baseline_em, fdsa_f1 - baseline_f1))


## 7. Direct verification — raw predictions side by side (first 10 examples)

In [ ]:
print(f"{'#':<3}{'BASELINE':<30}{'FDSA (margin=5.0)':<30}{'MATCH?':<6}")
print("-"*70)
for i in range(min(10, len(baseline_preds))):
    match = "YES" if baseline_preds[i].strip() == fdsa_preds[i].strip() else "NO"
    print(f"{i:<3}{baseline_preds[i][:28]:<30}{fdsa_preds[i][:28]:<30}{match:<6}")


## 8. If you want to see the filter actually bind — try a tight margin

margin=5.0 may still be loose enough to never affect beam-4 selection, same failure mode as
V2_U1. Run this with margin=1.0 or margin=0.5 to find where it starts changing predictions —
that's the point where you can start asking whether it helps or hurts accuracy, which is the
real, answerable question this architecture supports.


In [ ]:
for test_margin in [2.0, 1.0, 0.5]:
    preds, t, tps = run_generation(questions, make_kwargs(margin=test_margin))
    em = sum(exact_match(p, ex["answers"]) for p, ex in zip(preds, eval_set)) / len(eval_set)
    f1 = sum(f1_score(p, ex["answers"]) for p, ex in zip(preds, eval_set)) / len(eval_set)
    n_changed = sum(1 for b, p in zip(baseline_preds, preds) if b.strip() != p.strip())
    print(f"margin={test_margin:<5} EM: {em:.4f}  F1: {f1:.4f}  "
          f"predictions changed vs baseline: {n_changed}/{len(preds)}")
